# YOLO Webcam Demo for Google Colab

This notebook runs your trained YOLO model on frames captured from your laptop camera through the Colab browser session.

Run the cells from top to bottom:
1. Install dependencies.
2. Upload `best.pt` from your laptop.
3. Load the model.
4. Start the webcam helper.
5. Run the demo loop.
6. Stop the camera when finished.

Notes:
- In Colab, camera access happens through the browser, not through OpenCV directly.
- This is a simple demo loop, not low-latency realtime streaming.
- If your model file has a different name, update `MODEL_PATH` in the config cell.

## 1. Install YOLO and Colab Dependencies

This installs the model runtime, image utilities, and Colab widget support.

In [ ]:
%pip install -q ultralytics opencv-python-headless pillow ipywidgets

In [ ]:
import base64
import io
import os
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
from IPython.display import HTML, Javascript, clear_output, display

from google.colab import files
from google.colab.output import eval_js

## 2. Upload the Trained YOLO Weights

Upload your `best.pt` file from your laptop into the Colab runtime. If the uploaded file has a different name, update `MODEL_PATH` afterward.

In [ ]:
MODEL_PATH = 'best.pt'
IMAGE_SIZE = 960
CONFIDENCE = 0.10
IOU = 0.45
MAX_FRAMES = 120
FRAME_DELAY = 0.05
CLASS_FILTER = None  # Example: [0] for one class, or keep None for all classes.

uploaded = files.upload()
if uploaded:
    uploaded_names = list(uploaded.keys())
    print(f'Uploaded files: {uploaded_names}')
    if MODEL_PATH not in uploaded and len(uploaded_names) == 1:
        MODEL_PATH = uploaded_names[0]
        print(f"Using uploaded weights: {MODEL_PATH}")
else:
    print('No files uploaded in this step.')

model_file = Path(MODEL_PATH)
if not model_file.exists():
    raise FileNotFoundError(f'Model file not found: {model_file.resolve()}')

print(f'Model path ready: {model_file.resolve()}')
print({
    'IMAGE_SIZE': IMAGE_SIZE,
    'CONFIDENCE': CONFIDENCE,
    'IOU': IOU,
    'MAX_FRAMES': MAX_FRAMES,
    'FRAME_DELAY': FRAME_DELAY,
    'CLASS_FILTER': CLASS_FILTER,
})

## 3. Load the Model in Colab

This initializes the model and prints the detected class mapping from your trained weights.

In [ ]:
model = YOLO(MODEL_PATH)
print('Model loaded successfully.')
print('Classes:', model.names)
print('Tip: if no detections appear, lower CONFIDENCE to 0.05 and increase IMAGE_SIZE to 1280.')

## 4. Create a Browser Webcam Capture Helper

Colab can only access your laptop camera through the browser. This cell creates a small camera panel and JavaScript capture helpers.

In [ ]:
WEBCAM_HTML = '''
<div id="webcam-root" style="display:flex;gap:16px;align-items:flex-start;flex-wrap:wrap;">
  <div>
    <video id="webcam-video" width="640" height="480" autoplay playsinline style="border:1px solid #999;border-radius:8px;background:#000;"></video>
    <div style="margin-top:8px;display:flex;gap:8px;">
      <button onclick="startWebcam()">Start camera</button>
      <button onclick="stopWebcam()">Stop camera</button>
    </div>
  </div>
  <canvas id="webcam-canvas" width="640" height="480" style="display:none;"></canvas>
</div>
'''

WEBCAM_JS = '''
window.colabWebcamStream = window.colabWebcamStream || null;

async function startWebcam() {
  const video = document.getElementById('webcam-video');
  if (!video) {
    throw new Error('Webcam UI not found. Rerun the webcam helper cell first.');
  }
  if (window.colabWebcamStream) {
    return 'already-running';
  }
  const stream = await navigator.mediaDevices.getUserMedia({
    video: {
      width: { ideal: 1280 },
      height: { ideal: 720 },
      frameRate: { ideal: 30 }
    },
    audio: false
  });
  window.colabWebcamStream = stream;
  video.srcObject = stream;
  await video.play();
  return `${video.videoWidth}x${video.videoHeight}`;
}

async function captureWebcamFrame() {
  const video = document.getElementById('webcam-video');
  const canvas = document.getElementById('webcam-canvas');
  if (!video || !canvas) {
    throw new Error('Webcam UI not found. Rerun the webcam helper cell first.');
  }
  if (!window.colabWebcamStream) {
    throw new Error('Camera is not started. Click Start camera first.');
  }
  const context = canvas.getContext('2d');
  canvas.width = video.videoWidth || 1280;
  canvas.height = video.videoHeight || 720;
  context.drawImage(video, 0, 0, canvas.width, canvas.height);
  return canvas.toDataURL('image/jpeg', 0.95);
}

async function stopWebcam() {
  const video = document.getElementById('webcam-video');
  if (window.colabWebcamStream) {
    window.colabWebcamStream.getTracks().forEach(track => track.stop());
    window.colabWebcamStream = null;
  }
  if (video) {
    video.srcObject = null;
  }
  return 'stopped';
}
'''

def setup_webcam_ui() -> None:
    display(HTML(WEBCAM_HTML))
    display(Javascript(WEBCAM_JS))


setup_webcam_ui()
print('Camera helper ready. Click Start camera if prompted, allow access, and wait for the preview to appear.')

## 5. Run Inference on Webcam Frames

These helpers decode browser frames, run YOLO, and format the detections for display.

In [ ]:
LIVE_IMAGE_HANDLE = None
LIVE_STATUS_HANDLE = None


def decode_image(data_url: str) -> np.ndarray:
    encoded = data_url.split(',', 1)[1]
    image_bytes = base64.b64decode(encoded)
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)


def summarize_detections(result) -> list[str]:
    lines = []
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        label = result.names[class_id]
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        lines.append(f'{label}: {confidence:.2f} @ [{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]')
    return lines


def run_inference_on_frame(frame_bgr: np.ndarray):
    # Ultralytics handles numpy arrays well; we pass RGB to avoid color-space ambiguity.
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = model.predict(
        source=frame_rgb,
        conf=CONFIDENCE,
        iou=IOU,
        imgsz=IMAGE_SIZE,
        classes=CLASS_FILTER,
        max_det=100,
        verbose=False,
    )
    result = results[0]
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return result, Image.fromarray(annotated_rgb)


def display_result(frame_index: int, frame_shape: tuple[int, int, int], annotated_image: Image.Image, detections: list[str], latency_s: float) -> None:
    global LIVE_IMAGE_HANDLE, LIVE_STATUS_HANDLE

    status_lines = [
        f'Frame {frame_index} | Shape: {frame_shape[1]}x{frame_shape[0]}',
        f'Latency: {latency_s:.3f}s | Threshold: conf={CONFIDENCE}, iou={IOU}, imgsz={IMAGE_SIZE}',
        f'Detections count: {len(detections)}',
    ]
    if detections:
        status_lines.append('Detections:')
        status_lines.extend([f'  - {line}' for line in detections[:20]])
    else:
        status_lines.append('Detections: none')

    status_html = '<pre style="white-space:pre-wrap; font-family:monospace; margin-top:8px;">' + '\n'.join(status_lines) + '</pre>'

    if LIVE_IMAGE_HANDLE is None:
        LIVE_IMAGE_HANDLE = display(annotated_image, display_id=True)
        LIVE_STATUS_HANDLE = display(HTML(status_html), display_id=True)
    else:
        LIVE_IMAGE_HANDLE.update(annotated_image)
        LIVE_STATUS_HANDLE.update(HTML(status_html))

## 6. Draw Detection Boxes and Labels

Run this cell after clicking Start camera in the preview panel above.

## Optional: Single Image Sanity Check

If live webcam shows no detections, run the next code cell first. It verifies the model on one uploaded image and helps confirm whether the issue is model confidence or webcam setup.

In [ ]:
test_upload = files.upload()
if not test_upload:
    print('No image uploaded.')
else:
    test_name = list(test_upload.keys())[0]
    image = Image.open(io.BytesIO(test_upload[test_name])).convert('RGB')
    frame_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    result, annotated_image = run_inference_on_frame(frame_bgr)
    detections = summarize_detections(result)
    display(annotated_image)
    print(f'Test image: {test_name}')
    print(f'Detections count: {len(detections)}')
    for line in detections[:20]:
        print(f'  - {line}')

In [ ]:
print('If you have not already done it, click Start camera in the preview panel and allow browser access.')
print('Running live inference...')

try:
    global LIVE_IMAGE_HANDLE, LIVE_STATUS_HANDLE
    LIVE_IMAGE_HANDLE = None
    LIVE_STATUS_HANDLE = None

    helper_ready = eval_js("typeof captureWebcamFrame !== 'undefined'")
    if not helper_ready:
        print('Webcam helper was not active. Re-registering it now...')
        setup_webcam_ui()
        helper_ready = eval_js("typeof captureWebcamFrame !== 'undefined'")

    if not helper_ready:
        raise RuntimeError('Webcam helper is still unavailable. Rerun the webcam helper cell.')

    # Try to start camera automatically after helper reset.
    try:
        start_result = eval_js('startWebcam()')
        print(f'Camera start result: {start_result}')
    except Exception as start_error:
        raise RuntimeError(
            'Could not auto-start camera. Click Start camera in the webcam panel, allow permission, then rerun this cell.'
        ) from start_error

    for frame_index in range(1, MAX_FRAMES + 1):
        loop_start = time.time()
        frame_data = eval_js('captureWebcamFrame()')
        frame_bgr = decode_image(frame_data)
        result, annotated_image = run_inference_on_frame(frame_bgr)
        detections = summarize_detections(result)
        latency_s = time.time() - loop_start
        display_result(frame_index, frame_bgr.shape, annotated_image, detections, latency_s)
        time.sleep(FRAME_DELAY)
except KeyboardInterrupt:
    print('Demo interrupted by user.')
except Exception as error:
    print(f'Demo stopped: {error}')
finally:
    print('Demo loop finished.')

## 7. Tune Confidence and Image Size

Adjust these values to trade off speed and detection quality, then rerun the model cell if needed.

In [ ]:
CONFIDENCE = 0.25
IOU = 0.45
IMAGE_SIZE = 640
MAX_FRAMES = 50
FRAME_DELAY = 0.2

print({
    'CONFIDENCE': CONFIDENCE,
    'IOU': IOU,
    'IMAGE_SIZE': IMAGE_SIZE,
    'MAX_FRAMES': MAX_FRAMES,
    'FRAME_DELAY': FRAME_DELAY,
})

## 8. Stop the Demo and Release Resources

Use this when you are done, or if you want to restart the webcam helper cleanly.

In [ ]:
try:
    eval_js('stopWebcam()')
    clear_output(wait=False)
    print('Webcam stopped.')
except Exception as error:
    print(f'Cleanup note: {error}')
    print('If stopWebcam is undefined, rerun the webcam helper cell once before cleanup.')